In [1]:
import os
import sys

In [2]:
# this adds the module to the path so that it can be used to call the functions
root = os.path.abspath(os.path.join(os.path.dirname(os.getcwd())))
sys.path.append(root)

In [3]:
from src.classes import Node, Cirq_Tableau

In [9]:
def couple_dict_maker(coupling_list, layout_map):
    couple_dict = dict()
    for couple in coupling_list:
        if couple[0] not in layout_map.keys() and couple[1] not in layout_map.keys():
            continue
        elif couple[0] in layout_map.keys() and couple[1] not in layout_map.keys():
            ctrl = layout_map[couple[0]]
            couple_dict[ctrl] = [[],[]]
        elif couple[0] not in layout_map.keys() and couple[1] in layout_map.keys():
            targ = layout_map[couple[1]]
            couple_dict[targ] = [[],[]]
        else:
            ctrl = layout_map[couple[0]]
            targ = layout_map[couple[1]]
            cple = [ctrl,targ]
            if ctrl in couple_dict:
                couple_dict[ctrl][0].append(cple)
            else:
                couple_dict[ctrl] = [[cple],[]]

            if targ in couple_dict:
                couple_dict[targ][1].append(cple)
            else:
                couple_dict[targ] = [[],[cple]]

    return couple_dict

In [7]:
def least_connected(couple_dict):
    connections = float('inf')
    indexes = []
    
    for ndx in couple_dict.keys():
        new_connections = len(couple_dict[ndx][0]) + len(couple_dict[ndx][1])
        if new_connections < connections:
            connections = new_connections
            indexes.append(ndx)
        elif new_connections == connections:
            indexes.append(ndx)

    return indexes

In [50]:
[1,2] +[3]

[1, 2, 3]

In [ ]:
def generate_ops(tableau, coupling_dict, pauli_ndx, indexes):
    operations = []
    for ndx in indexes:
        if ~tab.xs[pauli_ndx][ndx] & ~tab.zs[pauli_ndx][ndx]: # index has an I already solved
            continue
        
        ctrl_list, targ_list = coupling_dict[ndx][0], coupling_dict[ndx][1]

        for couple in ctrl_list:
            op = []
            if ~tab.xs[pauli_ndx][couple[1]] & ~tab.zs[pauli_ndx][couple[1]]: # target is I
                operations.append([("CX", couple[0], couple[1]),("CX", couple[1], couple[0]),("CX", couple[0], couple[1])])
                continue
                
            if tab.xs[pauli_ndx][couple[0]] & ~tab.zs[pauli_ndx][couple[0]]: #control is X
                op.append(("H", couple[0]))
            elif tab.xs[pauli_ndx][couple[0]] & tab.zs[pauli_ndx][couple[0]]: #control is Y
                op.append(("S", couple[0]))
                op.append(("H", couple[0]))
                
            if tab.xs[pauli_ndx][couple[1]] & ~tab.zs[pauli_ndx][couple[1]]: #target is X
                operations.append(op + [("H", couple[1]), ("CX", couple[0], couple[1])])
                operations.append(op + [("S", couple[1]), ("CX", couple[0], couple[1])])
            else:
                operations.append(op + [("CX", couple[0], couple[1])])

        for couple in targ_list:
            op = []
            if ~tab.xs[pauli_ndx][couple[0]] & ~tab.zs[pauli_ndx][couple[0]]: # control is I
                operations.append([("CX", couple[0], couple[1]),("CX", couple[1], couple[0]),("CX", couple[0], couple[1])])
                continue
                
            if tab.xs[pauli_ndx][couple[1]] & tab.zs[pauli_ndx][couple[1]]: #target is Y
                op.append(("S", couple[1]))
            elif ~tab.xs[pauli_ndx][couple[1]] & tab.zs[pauli_ndx][couple[1]]: #target is Z
                op.append(("H", couple[1]))
                
            if ~tab.xs[pauli_ndx][couple[0]] & tab.zs[pauli_ndx][couple[0]]: #control is Z
                operations.append(op + [("H", couple[0]), ("CX", couple[0], couple[1])])
                operations.append(op + [("H", couple[0]),("S", couple[0]), ("CX", couple[0], couple[1])])
            else:
                operations.append(op + [("CX", couple[0], couple[1])])

    return operations

In [ ]:
def hrdwre_aware_solve(tableau, pauli_ndx, ndx_list, coupling_list, layout_map):
    tab = tableau.copy()
    reduction = []
    
    coupling_dict = couple_dict_maker(coupling_list)
    
    while True:
        operation = None
        benefit = -float('inf')
        reduced = 0
        increased = 0

        # break condition 
        if sum(tab.xs[ndx] | tab.zs[ndx]) == 1:
            break

        least_ndxs = least_connected(coupling_dict)
        operations = generate_ops(tab,coupling_dict,pauli_ndx, least_ndxs)
            